# Proyecto Capstone: Experiencia de Usuario en Games Colombia SAS
**Objetivo:** Segmentar perfiles de jugadores y clasificar el nivel de engagement utilizando técnicas de Machine Learning y Deep Learning (Redes Neuronales Densas).

---
## Contenidos del Notebook:
1. **Fase 1: Análisis Exploratorio de Datos (EDA)**
2. **Fase 2: Preprocesamiento e Ingeniería de Características**
3. **Fase 3: Segmentación No Supervisada (K-Means Clustering)**
4. **Fase 4: Modelo Baseline (Random Forest Classifier)**
5. **Fase 5: Clasificación con Red Neuronal Densa (MLP en Keras/TensorFlow)**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import tensorflow as tf
from tensorflow.keras import layers, models

# Configuración de visualización
sns.set_theme(style="whitegrid")
%matplotlib inline

## Fase 1: Análisis Exploratorio de Datos (EDA)
Cargaremos el dataset `online_gaming_behavior_dataset.csv` y analizaremos sus propiedades fundamentales.

In [ ]:
# Carga de datos
df = pd.read_csv("online_gaming_behavior_dataset.csv")
print(f"Dimensiones del dataset: {df.shape[0]} filas, {df.shape[1]} columnas")
df.head()

In [ ]:
# Resumen de información del dataset
df.info()

In [ ]:
# Estadísticas descriptivas de variables numéricas
df.describe()

In [ ]:
# Distribución de la variable objetivo (EngagementLevel)
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='EngagementLevel', order=['Low', 'Medium', 'High'], palette='viridis')
plt.title('Distribución de Niveles de Engagement')
plt.xlabel('Nivel de Engagement')
plt.ylabel('Cantidad de Jugadores')
plt.show()

In [ ]:
# Relación entre horas de juego y nivel de engagement
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='EngagementLevel', y='PlayTimeHours', order=['Low', 'Medium', 'High'], palette='Set2')
plt.title('Horas de Juego Promedio vs. Nivel de Engagement')
plt.xlabel('Nivel de Engagement')
plt.ylabel('Horas de Juego por Sesión')
plt.show()

## Fase 2: Preprocesamiento de Datos
En esta sección prepararemos las características para nuestros modelos:
1. Eliminar identificadores irrelevantes como `PlayerID`.
2. Convertir variables categóricas a numéricas mediante codificación One-Hot.
3. Codificar la variable objetivo `EngagementLevel` a valores numéricos (0 = Low, 1 = Medium, 2 = High).
4. Dividir los datos en conjuntos de entrenamiento (80%) y evaluación (20%).
5. Normalizar las características numéricas para la red neuronal.

In [ ]:
# 1. Eliminación de columna irrelevante
df_clean = df.drop(columns=['PlayerID'])

# 2. Codificación de variables categóricas (One-Hot Encoding)
categorical_cols = ['Gender', 'Location', 'GameGenre', 'GameDifficulty']
df_preprocessed = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True)

# 3. Codificación de la variable objetivo
target_map = {'Low': 0, 'Medium': 1, 'High': 2}
df_preprocessed['EngagementLevel'] = df_preprocessed['EngagementLevel'].map(target_map)

df_preprocessed.head()

In [ ]:
# Separación de características (X) y objetivo (y)
X = df_preprocessed.drop(columns=['EngagementLevel'])
y = df_preprocessed['EngagementLevel']

# División en train y test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Entrenamiento: X={X_train.shape}, y={y_train.shape}")
print(f"Evaluación: X={X_test.shape}, y={y_test.shape}")

In [ ]:
# Normalización de características numéricas con MinMaxScaler
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Fase 3: Segmentación No Supervisada (K-Means Clustering)
De acuerdo a los perfiles de comportamiento de consumo de la plataforma, agruparemos a los jugadores en segmentos independientes.

In [ ]:
# Utilizaremos K-Means para segmentar en 3 grupos basados en horas de juego, sesiones semanales, compras y logros
features_for_clustering = ['PlayTimeHours', 'SessionsPerWeek', 'AvgSessionDurationMinutes', 'InGamePurchases', 'AchievementsUnlocked']

# Escalamos estas características de forma independiente para el clustering
scaler_clust = StandardScaler()
X_clust = scaler_clust.fit_transform(df_clean[features_for_clustering])

# Ajustar el modelo de K-Means
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_clean['Cluster'] = kmeans.fit_predict(X_clust)

# Analizar las características promedio de cada cluster
cluster_profile = df_clean.groupby('Cluster')[features_for_clustering].mean()
cluster_profile['Count'] = df_clean['Cluster'].value_counts()
cluster_profile

### Interpretación de los Clústeres:
* **Cluster 0:** Jugadores ocasionales (bajo tiempo de juego, pocas sesiones).
* **Cluster 1:** Jugadores comprometidos / "Achievers" (muchas horas de juego, alto nivel de logros desbloqueados).
* **Cluster 2:** Jugadores monetizados / "Spenders" (alto índice de compras dentro del juego, sesiones regulares).
*(Nota: la asignación de números de clúster puede variar según la semilla aleatoria).*Lograremos perfilar estos grupos para diseñar campañas específicas.

## Fase 4: Modelo Baseline (Random Forest Classifier)
Crearemos un clasificador robusto clásico basado en árboles para predecir el nivel de engagement del jugador.

In [ ]:
# Entrenamiento del modelo Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Predicciones
y_pred_rf = rf_model.predict(X_test)

# Métricas de evaluación
print("--- MÉTRICAS RANDOM FOREST ---")
print(f"Exactitud (Accuracy): {accuracy_score(y_test, y_pred_rf):.4f}\n")
print(classification_report(y_test, y_pred_rf, target_names=['Low', 'Medium', 'High']))

In [ ]:
# Matriz de Confusión
plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Low', 'Medium', 'High'], yticklabels=['Low', 'Medium', 'High'])
plt.title('Matriz de Confusión - Random Forest')
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.show()

## Fase 5: Clasificación con Red Neuronal Densa (MLP en Keras/TensorFlow)
Implementaremos una red neuronal densa para predecir las probabilidades del nivel de engagement de acuerdo con la explicación del profesor.

In [ ]:
# Configuración del modelo secuencial
model_mlp = models.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(3, activation='softmax')  # Salida Softmax para las 3 clases
])

model_mlp.summary()

In [ ]:
# Compilación
model_mlp.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy', # Entrada y_train es numérica (0, 1, 2)
                  metrics=['accuracy'])

# Entrenamiento
history = model_mlp.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=64,
    verbose=1
)

In [ ]:
# Graficar pérdida e exactitud
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Entrenamiento')
plt.plot(history.history['val_loss'], label='Validación')
plt.title('Curva de Pérdida (Loss)')
plt.xlabel('Época')
plt.ylabel('Pérdida')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Entrenamiento')
plt.plot(history.history['val_accuracy'], label='Validación')
plt.title('Curva de Exactitud (Accuracy)')
plt.xlabel('Época')
plt.ylabel('Exactitud')
plt.legend()

plt.show()

In [ ]:
# Predicciones con la Red Neuronal
y_pred_probs = model_mlp.predict(X_test_scaled)
y_pred_mlp = np.argmax(y_pred_probs, axis=1) # Se selecciona la clase con mayor probabilidad

# Métricas de evaluación
print("--- MÉTRICAS RED NEURONAL DENSA (MLP) ---")
print(f"Exactitud (Accuracy): {accuracy_score(y_test, y_pred_mlp):.4f}\n")
print(classification_report(y_test, y_pred_mlp, target_names=['Low', 'Medium', 'High']))

In [ ]:
# Ejemplo de clasificación de un nuevo jugador (con probabilidades)
new_player_features = X_test_scaled[0:1]
probabilities = model_mlp.predict(new_player_features)[0]

print("Probabilidades por clase:")
print(f" - Bajo Engagement (Low): {probabilities[0]*100:.2f}%")
print(f" - Medio Engagement (Medium): {probabilities[1]*100:.2f}%")
print(f" - Alto Engagement (High): {probabilities[2]*100:.2f}%")

classes = ['Low', 'Medium', 'High']
assigned_class = classes[np.argmax(probabilities)]
print(f"\nEl jugador es asignado al grupo: {assigned_class} (con la mayor probabilidad)")